In [ ]:
import os

# Create a directory for papers if it doesn't exist
if not os.path.exists('papers'):
    os.makedirs('papers')

print("Please upload your PDF files to the 'papers' folder using the file browser on the left.")
print("After uploading, you can run the next cell to verify the files.")

Please upload your PDF files to the 'papers' folder using the file browser on the left.
After uploading, you can run the next cell to verify the files.


In [ ]:
import os

papers_dir = 'papers'

# List files in the papers directory
uploaded_files = os.listdir(papers_dir)

if uploaded_files:
    print(f"Found {len(uploaded_files)} files in '{papers_dir}':")
    for file_name in uploaded_files:
        print(f"- {file_name}")
else:
    print(f"No files found in '{papers_dir}'. Please upload your PDF files.")

No files found in 'papers'. Please upload your PDF files.


In [ ]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter # Corrected import path
import os

papers_dir = 'papers'
all_documents = []

print("Loading documents...")
for filename in os.listdir(papers_dir):
    if filename.endswith('.pdf'):
        file_path = os.path.join(papers_dir, filename)
        try:
            loader = PyPDFLoader(file_path)
            documents = loader.load()
            all_documents.extend(documents)
            print(f"Loaded {len(documents)} pages from {filename}")
        except Exception as e:
            print(f"Error loading {filename}: {e}")

if not all_documents:
    print("No documents were loaded. Please ensure you have uploaded PDF files to the 'papers' directory.")
else:
    print(f"Total pages loaded: {len(all_documents)}")

    # Split documents into chunks
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = 1000, # Max size of each chunk
        chunk_overlap  = 100, # Overlap between chunks to maintain context
        length_function = len,
        is_separator_regex = False,
    )

    chunks = text_splitter.split_documents(all_documents)
    print(f"Split {len(all_documents)} pages into {len(chunks)} chunks.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


/tmp/ipykernel_567/1187588278.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loading documents...
No documents were loaded. Please ensure you have uploaded PDF files to the 'papers' directory.


In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings # Updated import path
from langchain_chroma import Chroma # Updated import path

# Initialize the embedding model
# You can choose different models from Hugging Face based on your needs.
# 'all-MiniLM-L6-v2' is a good balance of performance and size.
print("Initializing embedding model...")
embedding_model_name = "all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)
print(f"Embedding model '{embedding_model_name}' initialized.")

# Create a Chroma vector store from the document chunks and embeddings
# This will persist the database to disk in the 'chroma_db' directory
print("Creating and persisting ChromaDB vector store...")

# Check if 'chunks' is defined and not empty before proceeding
if 'chunks' in locals() and chunks:
    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory="chroma_db"
    )
    vector_store.persist()
    print("ChromaDB vector store created and persisted to 'chroma_db/'.")
    print(f"Number of embeddings in vector store: {vector_store._collection.count()}")
else:
    print("Cannot create vector store: 'chunks' not found or is empty. Please ensure you have uploaded PDF files and successfully run the 'Load and Process PDFs' cell.")
    vector_store = None # Ensure vector_store is defined as None if not created

Initializing embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model 'all-MiniLM-L6-v2' initialized.
Creating and persisting ChromaDB vector store...
Cannot create vector store: 'chunks' not found or is empty. Please ensure you have uploaded PDF files and successfully run the 'Load and Process PDFs' cell.


In [ ]:
import google.generativeai as genai
from langchain_google_genai import GoogleGenerativeAI
from langchain.chains import RetrievalQA # This import should now be correct as 'langchain' package is installed
from IPython.display import display, Markdown
import os

# Configure Google Generative AI
# Ensure you have your GOOGLE_API_KEY set up as an environment variable
# You can also set it directly: os.environ["GOOGLE_API_KEY"] = "YOUR_API_KEY"

# For Colab, you can get the API key from the 'Secrets' tab on the left sidebar
# Then add it to your environment variables
from google.colab import userdata

try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)
    print("Google API Key loaded from Colab Secrets.")
except Exception as e:
    print(f"Error loading Google API Key from Colab Secrets: {e}")
    print("Please ensure 'GOOGLE_API_KEY' is set in Colab Secrets or as an environment variable.")

# Initialize the LLM
print("Initializing Google Generative AI LLM...")
llm = GoogleGenerativeAI(model="gemini-pro", temperature=0.2)
print("LLM initialized.")

# Create a retriever from the vector store
print("Setting up retriever...")
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3} # Retrieve top 3 most similar documents
)
print("Retriever set up.")

# Create the RAG chain
print("Building RetrievalQA chain...")
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff", # 'stuff' chain type concatenates all documents into a single prompt
    retriever=retriever,
    return_source_documents=True
)
print("RetrievalQA chain built.")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


ModuleNotFoundError: No module named 'langchain.chains'

In [ ]:
# Example Query
query = "What is the main idea behind Transformer models?"
print(f"Query: {query}\n")

response = qa_chain({"query": query})

# Display the answer and sources
display(Markdown(f"**Answer:** {response['result']}"))
display(Markdown("**Sources:**"))
for i, doc in enumerate(response['source_documents']):
    display(Markdown(f"- Document {i+1}: {doc.metadata.get('source', 'N/A')} (Page: {doc.metadata.get('page', 'N/A')})"))
    # Optionally display a snippet of the retrieved content:
    # display(Markdown(f"  Snippet: {doc.page_content[:200]}..."))

# You can continue asking more questions by modifying the 'query' variable above and re-running this cell.

Query: What is the main idea behind Transformer models?



NameError: name 'qa_chain' is not defined